# 07a — Retrieval from first principles — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/07.md) · [Course map](../PLAN.md) · [Project](../../projects/stage07/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Chunk documents with stable provenance
- Rank keyword matches
- Measure retrieval separately from answers


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Retrieval as a measurable search pipeline

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

Retrieval finds evidence before answering. A document is split into chunks so useful passages fit a search/model budget. Each chunk needs source identity, location, content version, and access scope. Chunk size and overlap change whether relevant facts stay together.

Lexical retrieval compares words. BM25 accounts for term frequency, document length, and how common a term is across documents. Dense retrieval compares learned embedding vectors. Hybrid retrieval combines signals; reciprocal-rank fusion adds contributions based on positions in result lists. A reranker rescoring candidates cannot recover a document absent from its candidate set.

Recall@k measures how much labeled relevant evidence appears among the first k results. MRR rewards an early first relevant result using reciprocal rank. Use the same labeled queries to compare methods.

### Worked example: follow the values

If the relevant sources are A and C and the top three results are B,A,D, recall@3 is 1/2. The first relevant result is at rank 2, so reciprocal rank is 1/2. If no relevant result appears, reciprocal rank is zero. Define separate behavior for questions with no relevant sources.

### Pause and explain

Why store a content hash alongside a chunk?

<details><summary>Check your reasoning after trying</summary>

It identifies the version of text that produced the chunk or vector, helping detect changed sources and stale indexes.

</details>

### Common mistakes to check

Check authorization before building candidates. Word offsets and character offsets are different units; document which one you store.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

Retrieval selects evidence before an answer is generated. Parsing converts PDF, Markdown, HTML, DOCX, JSON, or database records into text and metadata. Preserve document ID, page or section, source version, tenant, and permissions. OCR and tables need special care; a parser can silently scramble numbers.

Chunking balances context completeness against retrieval precision. Fixed-size chunks are simple; paragraph and semantic chunks follow content boundaries. Overlap can preserve context but also duplicates evidence and consumes space. Use the model tokenizer for actual token limits; the exercise uses whitespace words as a transparent baseline.

Keyword methods match literal terms. BM25 adjusts term frequency using document length and rarity. Dense embeddings can retrieve paraphrases but may miss exact identifiers. Hybrid retrieval combines complementary ranked lists; reranking scores query-candidate pairs with a more expensive model. First establish a measured lexical baseline so you can tell whether embeddings add value.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import re
print(re.findall(r"[a-z0-9]+", "Revenue grew 12% in 2025.".lower()))


## Exercise 1: Overlapping chunks

Return word chunks with size>0 and 0<=overlap<size. Avoid an extra final chunk made only of overlap.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def chunks(text, size=4, overlap=1):
    if size <= 0 or not 0 <= overlap < size: raise ValueError("Invalid chunk configuration")
    words, result = text.split(), []
    start = 0
    while start < len(words):
        result.append(" ".join(words[start:start+size]))
        if start+size >= len(words): break
        start += size-overlap
    return result


In [ ]:
assert chunks("a b c d e f", 4, 1) == ["a b c d", "d e f"]
assert chunks("") == []
expect_error(ValueError, lambda: chunks("a", 2, 2))
print("Exercise 1: checks passed")


**Why this works:** Poor stopping conditions create redundant tail chunks that skew ranking.


## Exercise 2: Lexical retrieval

For dicts with id,text, rank documents by number of shared unique lowercase alphanumeric terms; omit zero scores, break ties by id.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def retrieve(query, documents, k=3):
    import re
    terms = lambda s: set(re.findall(r"[a-z0-9]+", s.lower()))
    q = terms(query)
    scored = [(len(q & terms(d["text"])), d["id"]) for d in documents]
    return [id for score, id in sorted(scored, key=lambda pair: (-pair[0], pair[1])) if score > 0][:k]


In [ ]:
docs = [{"id": "a", "text": "Revenue margin"}, {"id": "b", "text": "Password reset"}]
assert retrieve("revenue", docs) == ["a"]
assert retrieve("unicorn", docs) == []
print("Exercise 2: checks passed")


**Why this works:** This is an inspectable baseline, not semantic retrieval or BM25.


## Exercise 3: Reciprocal rank fusion

Combine ranked ID lists using sum(1/(c+rank)), rank starting at 1. Count an ID at most once per list; tie-break by ID.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def rrf(rankings, c=60):
    scores = {}
    for ranking in rankings:
        seen = set()
        for rank, id in enumerate(ranking, 1):
            if id not in seen:
                scores[id] = scores.get(id, 0)+1/(c+rank)
                seen.add(id)
    return sorted(scores, key=lambda id: (-scores[id], id))


In [ ]:
assert rrf([["a", "b"], ["b", "c"]])[0] == "b"
print("Exercise 3: checks passed")


**Why this works:** Raw cosine and keyword scores are not directly comparable without calibration.


## Independent transfer

Build a private document index with chunk IDs, source offsets, parser version, and content hashes. Compare keyword, embedding, hybrid, and reranked recall on the same labeled questions.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [07a interface and acceptance cases](../assignments/07a.md).
2. Copy the [blank starter](../assignments/starters/07a.py) to `work/assignments/07a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 07a --solution work/assignments/07a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/07a.md#07a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 07a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why keep source offsets?

   To verify citations and trace a parsing or chunking error to the original document.

2. What can reranking fix?

   Ordering among retrieved candidates; it cannot recover evidence that never entered the candidate set.


## Reading and review

- [Primary reference 1](https://github.com/pgvector/pgvector)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
